# Standard Transformers model comparison

This notebook evaluates one selected standard Hugging Face causal language model against the 12 prompt instances in prompt_configurations.json.

The default run processes one prompt instance. Change RUN_LIMIT to None after the smoke test to run all 12 instances. Only the selected model is loaded.

## 1. Install dependencies

Run this cell once in the Colab runtime.

In [ ]:
!pip -q install transformers accelerate bitsandbytes huggingface_hub sentencepiece safetensors

## 2. Imports, optional Hugging Face login, and cache locations

Set HF_TOKEN in Colab secrets or the environment when a model requires access approval. Do not write the token into an output file.

In [ ]:
import gc
import json
import os
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

MODEL_CACHE = Path("/content/hf_cache")
OUTPUT_ROOT = Path("/content/experiment_outputs/model_comparison")
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---------- 4-bit, and GPU setup ----------
%pip install -q bitsandbytes
import gc
import torch

def clear_gpu_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

clear_gpu_cache()

## 3. Upload and load the prompt configuration

Upload the experimental_setup/prompt_configurations.json file when it is not already present in /content.

In [ ]:
from google.colab import files

CONFIG_PATH = Path("/content/prompt_configurations.json")
if not CONFIG_PATH.exists():
    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())
    if not uploaded_names:
        raise FileNotFoundError("Upload prompt_configurations.json before continuing.")
    CONFIG_PATH = Path("/content") / uploaded_names[0]

PROMPT_CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
PROMPT_INSTANCES = PROMPT_CONFIG["prompt_instances"]
assert len(PROMPT_INSTANCES) == 12, "Expected 12 prompt instances."
print("Loaded:", CONFIG_PATH)
print("Prompt instances:", len(PROMPT_INSTANCES))

## 4. Select the model and run size

Change only MODEL_KEY to compare another standard model. The notebook does not download the other models.

In [ ]:
MODEL_REGISTRY = {
    "goasty_qwen3_4b_indian_law": {
        "model_id": "goasty/Qwen3-4B-Indian-Law",
        "family": "Qwen3",
    },
    "qwen3_8b_base": {
        "model_id": "Qwen/Qwen3-8B-Base",
        "family": "Qwen3-base",
    },
    "qwen2_5_7b_instruct": {
        "model_id": "Qwen/Qwen2.5-7B-Instruct",
        "family": "Qwen2.5",
    },
    "mistral_7b_v03": {
        "model_id": "mistralai/Mistral-7B-v0.3",
        "family": "Mistral",
    },
    "saul_7b_instruct": {
        "model_id": "Equall/Saul-7B-Instruct-v1",
        "family": "Saul",
    },
    "adaptllm_law_llm": {
        "model_id": "AdaptLLM/law-LLM",
        "family": "AdaptLLM",
    },
}

MODEL_KEY = "saul_7b_instruct"
RUN_LIMIT = 1                 # Set to None to process all 12 instances.
PROMPT_INSTANCE_IDS = None    # Or provide a list of IDs for a custom subset.
MAX_NEW_TOKENS = 1024
USE_4BIT = True
RUN_EXPERIMENT = True

if MODEL_KEY not in MODEL_REGISTRY:
    raise KeyError(f"Unknown MODEL_KEY: {MODEL_KEY}")
MODEL_SPEC = MODEL_REGISTRY[MODEL_KEY]

## 5. Build prompts from the JSON configuration

The prompt order is fixed: instruction, allowed labels, schema, and document.

In [ ]:
def render_allowed_labels(labels):
    lines = []
    for item in labels:
        if isinstance(item, dict):
            lines.append(f"- {item['label']}: {item['description']}")
        else:
            lines.append(f"- {item}")
    return "\n".join(lines)


def render_schema(schema):
    if isinstance(schema, dict):
        return json.dumps(schema, ensure_ascii=False)
    return str(schema)


def build_prompt(instance):
    prompt_set = PROMPT_CONFIG["prompt_sets"][instance["prompt_set"]]
    document = instance["document"]["text"]
    return (
        f"{prompt_set['instruction']}\n\n"
        f"Allowed labels:\n{render_allowed_labels(prompt_set['allowed_labels'])}\n\n"
        f"Schema:\n{render_schema(prompt_set['schema'])}\n\n"
        f"DOCUMENT:\n{document}\n\n"
        "OUTPUT:"
    )


def select_instances():
    instances = PROMPT_INSTANCES
    if PROMPT_INSTANCE_IDS is not None:
        wanted = set(PROMPT_INSTANCE_IDS)
        instances = [item for item in instances if item["id"] in wanted]
    if RUN_LIMIT is not None:
        instances = instances[:RUN_LIMIT]
    if not instances:
        raise ValueError("No prompt instances selected.")
    return instances


print(build_prompt(select_instances()[0])[:2000])

## 6. Load, generate, clean up, and save raw outputs

In [ ]:
def model_input_device(model):
    for parameter in model.parameters():
        if parameter.device.type != "meta":
            return parameter.device
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_selected_model():
    quantization_config = None
    if USE_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    kwargs = {
        "device_map": "auto",
        "cache_dir": str(MODEL_CACHE / MODEL_KEY),
        "torch_dtype": torch.float16,
    }
    if quantization_config is not None:
        kwargs["quantization_config"] = quantization_config

    model_id = MODEL_SPEC["model_id"]
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        cache_dir=str(MODEL_CACHE / MODEL_KEY),
    )
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.eval()
    return tokenizer, model


def prepare_inputs(tokenizer, prompt, device):
    chat_template = getattr(tokenizer, "chat_template", None)
    if chat_template:
        messages = [{"role": "user", "content": prompt}]
        try:
            encoded = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            )
        except (TypeError, ValueError):
            encoded = tokenizer(prompt, return_tensors="pt")
            return encoded.to(device), "plain_text_fallback"
        if hasattr(encoded, "to"):
            encoded = encoded.to(device)
        return encoded, "chat_template"
    return tokenizer(prompt, return_tensors="pt").to(device), "plain_text"


def generate_one(tokenizer, model, prompt):
    device = model_input_device(model)
    inputs, prompt_mode = prepare_inputs(tokenizer, prompt, device)
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    input_length = inputs["input_ids"].shape[-1]
    raw_output = tokenizer.decode(
        output[0][input_length:],
        skip_special_tokens=True,
    )
    return raw_output, time.perf_counter() - started, prompt_mode


def save_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def unload_model(model, tokenizer):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def run_experiment():
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    output_path = OUTPUT_ROOT / MODEL_KEY / f"{run_id}.json"
    selected = select_instances()
    payload = {
        "status": "running",
        "run_id": run_id,
        "model_key": MODEL_KEY,
        "model_id": MODEL_SPEC["model_id"],
        "loading_mode": "transformers",
        "quantization": "4bit_nf4" if USE_4BIT else "none",
        "prompt_config_path": str(CONFIG_PATH),
        "selected_instance_ids": [item["id"] for item in selected],
        "run_limit": RUN_LIMIT,
        "max_new_tokens": MAX_NEW_TOKENS,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": [],
    }
    save_json(output_path, payload)

    tokenizer = None
    model = None
    try:
        tokenizer, model = load_selected_model()
        payload["device"] = str(model_input_device(model))
        for instance in selected:
            prompt = build_prompt(instance)
            record = {
                "instance_id": instance["id"],
                "prompt_set": instance["prompt_set"],
                "source_type": instance["document"]["source_type"],
                "document_word_count": instance["document"]["actual_word_count"],
                "prompt": prompt,
                "raw_output": None,
                "prompt_mode": None,
                "seconds": None,
                "status": "error",
                "error": None,
            }
            try:
                raw, seconds, prompt_mode = generate_one(tokenizer, model, prompt)
                record.update(
                    raw_output=raw,
                    prompt_mode=prompt_mode,
                    seconds=seconds,
                    status="completed",
                )
            except Exception as exc:
                record["error"] = f"{type(exc).__name__}: {exc}"
            payload["records"].append(record)
            save_json(output_path, payload)
    except Exception as exc:
        payload["load_error"] = f"{type(exc).__name__}: {exc}"
    finally:
        if model is not None:
            unload_model(model, tokenizer)
    payload["status"] = "completed" if "load_error" not in payload else "load_error"
    save_json(output_path, payload)
    print(f"Saved: {output_path}")
    print("Completed records:", sum(r["status"] == "completed" for r in payload["records"]))
    return payload


if RUN_EXPERIMENT:
    result = run_experiment()